In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df= pd.read_csv('/content/rapido_july2025_data.csv')
print("Shape:", df.shape)
df.head()

Shape: (30000, 25)


,Booking_ID,Booking_Status,Booking_Value,Customer_ID,Driver_ID,Pickup_Location,Drop_Location,Ride_Distance(km),Ride_Time(min),Date,...,Driver_Rating,Canceled_Rides_by_Customer,Canceled_Rides_by_Driver,Incomplete_Rides,Incomplete_Rides_Reason,Total_Bookings,Canceled_Bookings,Canceled_Percentage,V_TAT,C_TAT
0,RAP20250700001,Completed,170.39,CUST_2824,DR_902,Pune,Delhi,2.74,42,2025-07-02,...,4.3,0,0,0,NaN,998,122,12.22,19,27
1,RAP20250700002,Incomplete,131.04,CUST_1409,DR_915,Chennai,Hyderabad,22.15,20,2025-07-09,...,4.9,0,0,1,Driver delayed,986,119,12.07,25,12
2,RAP20250700003,Completed,242.19,CUST_5506,DR_938,Delhi,Bengaluru,11.95,46,2025-07-13,...,4.7,0,0,0,NaN,972,114,11.73,18,25
3,RAP20250700004,Completed,78.18,CUST_5012,DR_213,Pune,Hyderabad,9.08,24,2025-07-27,...,4.0,0,0,0,NaN,961,112,11.65,11,6
4,RAP20250700005,Completed,159.33,CUST_4657,DR_783,Chennai,Bengaluru,22.97,40,2025-07-22,...,3.0,0,0,0,NaN,963,115,11.94,22,25


In [ ]:
print("COLUMN INFORMATION")
print(df.info())

COLUMN INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Booking_ID                  30000 non-null  object 
 1   Booking_Status              30000 non-null  object 
 2   Booking_Value               30000 non-null  float64
 3   Customer_ID                 30000 non-null  object 
 4   Driver_ID                   30000 non-null  object 
 5   Pickup_Location             30000 non-null  object 
 6   Drop_Location               30000 non-null  object 
 7   Ride_Distance(km)           30000 non-null  float64
 8   Ride_Time(min)              30000 non-null  int64  
 9   Date                        30000 non-null  object 
 10  Time                        30000 non-null  object 
 11  Vehicle_Type                30000 non-null  object 
 12  Vehicle_Image               30000 non-null  object 
 13  Payment_Meth

In [ ]:
print("NULL COUNTS")
print(df.isnull().sum())

NULL COUNTS
Booking_ID                        0
Booking_Status                    0
Booking_Value                     0
Customer_ID                       0
Driver_ID                         0
Pickup_Location                   0
Drop_Location                     0
Ride_Distance(km)                 0
Ride_Time(min)                    0
Date                              0
Time                              0
Vehicle_Type                      0
Vehicle_Image                     0
Payment_Method                    0
Customer_Rating                   0
Driver_Rating                     0
Canceled_Rides_by_Customer        0
Canceled_Rides_by_Driver          0
Incomplete_Rides                  0
Incomplete_Rides_Reason       28200
Total_Bookings                    0
Canceled_Bookings                 0
Canceled_Percentage               0
V_TAT                             0
C_TAT                             0
dtype: int64


In [ ]:
print("DUPLICATE Booking_IDs:", df["Booking_ID"].duplicated().sum())

DUPLICATE Booking_IDs: 0


In [ ]:
print("Booking_Status Values:", df["Booking_Status"].value_counts())

Booking_Status Values: Booking_Status
Completed     24641
Cancelled      3559
Incomplete     1800
Name: count, dtype: int64


In [ ]:
print("Vehicle_Type Values:", df["Vehicle_Type"].value_counts())

Vehicle_Type Values: Vehicle_Type
Bike    21001
Auto     8999
Name: count, dtype: int64


Data Quality Validation Checks

In [ ]:
# Check 1: cancellation flags should be consistent with Booking_Status
mismatch = df[(df['Booking_Status'] == 'Completed') &
              ((df['Canceled_Rides_by_Customer'] == 1) | (df['Canceled_Rides_by_Driver'] == 1))]
print("Completed rides wrongly flagged as cancelled:", len(mismatch))

# Check 2: confirm the pre-aggregated daily columns match your own rollup
daily_check = df.groupby('Date').size().reset_index(name='my_count')
daily_check = daily_check.merge(df[['Date','Total_Bookings']].drop_duplicates(), on='Date')
daily_check['match'] = daily_check['my_count'] == daily_check['Total_Bookings']
print("\nDaily rollup mismatches:", (~daily_check['match']).sum())

# Check 3: confirm fare vs distance correlation (already found ~0, re-verify)
completed = df[df['Booking_Status'] == 'Completed']
print("\nFare-distance correlation:", completed['Ride_Distance(km)'].corr(completed['Booking_Value']))

Completed rides wrongly flagged as cancelled: 0

Daily rollup mismatches: 0

Fare-distance correlation: 0.006490737742118286


Clean & Standardize

In [ ]:
df_clean = df.copy()

# Parse date/time properly
df_clean['Date'] = pd.to_datetime(df_clean['Date'])
df_clean['datetime'] = pd.to_datetime(df_clean['Date'].astype(str) + ' ' + df_clean['Time'].astype(str))

# Standardize text categories (safety net even if already clean)
for col in ['Vehicle_Type', 'Payment_Method', 'Booking_Status', 'Pickup_Location', 'Drop_Location']:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

# Drop the row-level pre-aggregated columns (they're daily snapshots, not per-ride facts)
df_clean = df_clean.drop(columns=['Total_Bookings', 'Canceled_Bookings', 'Canceled_Percentage', 'Vehicle_Image'])

print(df_clean.shape)
df_clean.head()

(30000, 22)


,Booking_ID,Booking_Status,Booking_Value,Customer_ID,Driver_ID,Pickup_Location,Drop_Location,Ride_Distance(km),Ride_Time(min),Date,...,Payment_Method,Customer_Rating,Driver_Rating,Canceled_Rides_by_Customer,Canceled_Rides_by_Driver,Incomplete_Rides,Incomplete_Rides_Reason,V_TAT,C_TAT,datetime
0,RAP20250700001,Completed,170.39,CUST_2824,DR_902,Pune,Delhi,2.74,42,2025-07-02,...,Upi,4.0,4.3,0,0,0,NaN,19,27,2025-07-02 09:43:00
1,RAP20250700002,Incomplete,131.04,CUST_1409,DR_915,Chennai,Hyderabad,22.15,20,2025-07-09,...,Cash,3.3,4.9,0,0,1,Driver delayed,25,12,2025-07-09 17:34:00
2,RAP20250700003,Completed,242.19,CUST_5506,DR_938,Delhi,Bengaluru,11.95,46,2025-07-13,...,Card,4.1,4.7,0,0,0,NaN,18,25,2025-07-13 23:54:00
3,RAP20250700004,Completed,78.18,CUST_5012,DR_213,Pune,Hyderabad,9.08,24,2025-07-27,...,Wallet,3.2,4.0,0,0,0,NaN,11,6,2025-07-27 17:41:00
4,RAP20250700005,Completed,159.33,CUST_4657,DR_783,Chennai,Bengaluru,22.97,40,2025-07-22,...,Cash,3.2,3.0,0,0,0,NaN,22,25,2025-07-22 22:52:00


Feature Engineering

In [ ]:
df_clean['day_of_week'] = df_clean['datetime'].dt.day_name()
df_clean['is_weekend'] = df_clean['datetime'].dt.dayofweek >= 5
df_clean['hour'] = df_clean['datetime'].dt.hour
df_clean['is_peak_hour'] = df_clean['hour'].between(7,10) | df_clean['hour'].between(17,21)
df_clean['week_of_month'] = ((df_clean['datetime'].dt.day - 1) // 7) + 1

df_clean['cancel_side'] = np.select(
    [df_clean['Canceled_Rides_by_Customer'] == 1, df_clean['Canceled_Rides_by_Driver'] == 1],
    ['Customer', 'Driver'],
    default='None'
)

zone_pair = df_clean['Pickup_Location'] + ' → ' + df_clean['Drop_Location']
df_clean['zone_pair'] = zone_pair

df_clean.head()

,Booking_ID,Booking_Status,Booking_Value,Customer_ID,Driver_ID,Pickup_Location,Drop_Location,Ride_Distance(km),Ride_Time(min),Date,...,V_TAT,C_TAT,datetime,day_of_week,is_weekend,hour,is_peak_hour,week_of_month,cancel_side,zone_pair
0,RAP20250700001,Completed,170.39,CUST_2824,DR_902,Pune,Delhi,2.74,42,2025-07-02,...,19,27,2025-07-02 09:43:00,Wednesday,False,9,True,1,None,Pune → Delhi
1,RAP20250700002,Incomplete,131.04,CUST_1409,DR_915,Chennai,Hyderabad,22.15,20,2025-07-09,...,25,12,2025-07-09 17:34:00,Wednesday,False,17,True,2,None,Chennai → Hyderabad
2,RAP20250700003,Completed,242.19,CUST_5506,DR_938,Delhi,Bengaluru,11.95,46,2025-07-13,...,18,25,2025-07-13 23:54:00,Sunday,True,23,False,2,None,Delhi → Bengaluru
3,RAP20250700004,Completed,78.18,CUST_5012,DR_213,Pune,Hyderabad,9.08,24,2025-07-27,...,11,6,2025-07-27 17:41:00,Sunday,True,17,True,4,None,Pune → Hyderabad
4,RAP20250700005,Completed,159.33,CUST_4657,DR_783,Chennai,Bengaluru,22.97,40,2025-07-22,...,22,25,2025-07-22 22:52:00,Tuesday,False,22,False,4,None,Chennai → Bengaluru


Build the Star Schema Tables

In [ ]:
# fact_bookings: keep FKs + numeric measures only
fact_bookings = fact_bookings.rename(columns={
    'Booking_ID': 'booking_id', 'Customer_ID': 'customer_id', 'Driver_ID': 'driver_id',
    'Vehicle_Type': 'vehicle_type', 'Pickup_Location': 'pickup_location', 'Drop_Location': 'drop_location',
    'Payment_Method': 'payment_method', 'Date': 'date', 'Booking_Status': 'booking_status',
    'Booking_Value': 'booking_value', 'Ride_Distance(km)': 'ride_distance', 'Ride_Time(min)': 'ride_time',
    'Customer_Rating': 'customer_rating', 'Driver_Rating': 'driver_rating',
    'V_TAT': 'v_tat', 'C_TAT': 'c_tat', 'Incomplete_Rides_Reason': 'incomplete_rides_reason'
})

fact_bookings = fact_bookings.drop(columns=['day_of_week', 'is_weekend', 'is_peak_hour', 'week_of_month'], errors='ignore')

dim_customers = dim_customers.rename(columns={'Customer_ID': 'customer_id'})
dim_drivers = dim_drivers.rename(columns={'Driver_ID': 'driver_id'})
dim_date = dim_date.rename(columns={'Date': 'date'})
dim_location = dim_location.rename(columns={'Pickup_Location': 'pickup_location', 'Drop_Location': 'drop_location'})


Export Everything

In [ ]:
fact_bookings['hour'] = df_clean['hour']
fact_bookings.to_csv('fact_bookings.csv', index=False)

In [ ]:
fact_bookings.to_csv('fact_bookings.csv', index=False)
dim_customers.to_csv('dim_customers.csv', index=False)
dim_drivers.to_csv('dim_drivers.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)
dim_location.to_csv('dim_location.csv', index=False)
df_clean.to_csv('rapido_cleaned_full.csv', index=False)

print("All files exported.")

All files exported.


In [ ]:
from google.colab import files
for f in ['fact_bookings.csv','dim_customers.csv','dim_drivers.csv','dim_date.csv','dim_location.csv','rapido_cleaned_full.csv']:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>